# E4 — 5-fold CV trên cấu hình đã thắng

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

Fold 1 đã xong (macro-F1 **0.7001**, WORKLOG S-069). Notebook này chạy **các fold còn
lại** của **đúng cấu hình đó**, không đổi gì khác.

**Vì sao phải làm việc này trước khi thử thêm ý tưởng mới:**

1. **CI hiện quá rộng để kết luận.** Fold 1 có 82 ca, CI95 của macro-F1 là
   `[0.597, 0.789]` — rộng 0.19. Gộp out-of-fold cho **394 ca**, CI hẹp lại khoảng
   một nửa. Không có con số này thì không có bảng kết quả nào báo cáo được.
2. **Toàn bộ đóng góp headline của đề tài cần 5 model.** Bất định *epistemic* =
   mức bất đồng giữa các thành viên ensemble. Một model đơn lẻ **không** đo được nó.
   Không có 5 fold thì không có deep ensemble, không có selective prediction đúng nghĩa.
3. **`macro-F1 @ coverage` không dùng được ở n=82** — lớp hiếm rơi xuống 1–2 ca khi
   cắt coverage. Ở 394 ca thì dùng được.

**Ngân sách:** ~3.9h/fold (đo ở fold 1: 45.9 giây/epoch × 300). Session Kaggle tối đa
12h ⇒ chạy **2 fold mỗi session**, hai session là xong. `resume: true` nên session bị
ngắt giữa chừng thì chạy lại chính cell train, không mất tiến trình.

## 0. Bootstrap

Dòng `repo commit` là bằng chứng đang chạy đúng bản code nào. Không khớp commit mới
nhất trên GitHub thì mọi thứ bên dưới vô nghĩa.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"
EXPERIMENT = "E4_cv"

# ---- THAM SỐ DUY NHẤT CẦN SỬA GIỮA HAI SESSION ----------------------------
FOLDS = [4, 5]          # session trước đã chạy [2, 3]; fold 1 từ notebook 06
# ---------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

# Python giữ module đã import trong sys.modules; clone lại KHÔNG tự làm mới (S-035).
for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

CACHE_DIR = Path("/kaggle/working/cache_e4")
os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)
os.environ["LLDMMRI_OUTPUT_DIR"] = f"/kaggle/working/runs/{EXPERIMENT}"
os.environ.pop("LLDMMRI_DATA_ROOT", None)   # env sót từ lần chạy trước sẽ đè config

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / "baseline_3dpatch.yaml"
CFG = load_yaml(CFG_PATH)
print("\nfold sẽ chạy trong session này:", FOLDS)
print("output dir:", os.environ["LLDMMRI_OUTPUT_DIR"])

# PHẦN A — Cache E4

Cache E4 là `lesion_tight · 112×112×32 · align_phases=per_phase`. Nếu đã upload nó
thành Kaggle Dataset thì mount vào (nhanh); nếu chưa thì build lại (~26 phút).

Build lại **cho ra đúng cùng dữ liệu** — pipeline tiền xử lý tất định, không có
randomness nào (`set_seed` chỉ ảnh hưởng train). Nên hai đường đều hợp lệ.

In [ ]:
import shutil

# Tìm cache E4 trong các dataset đã attach. Đổi/ thêm ứng viên nếu mount chỗ khác.
CANDIDATES = [
    Path("/kaggle/input/lld-mmri-e4-per-phase"),
    Path("/kaggle/input/lld-mmri-e4"),
]

mounted = None
for cand in CANDIDATES:
    if (cand / "cache_meta.json").exists():
        mounted = cand
    elif cand.exists():
        # dataset đôi khi bọc thêm một lớp thư mục
        for sub in cand.iterdir():
            if (sub / "cache_meta.json").exists():
                mounted = sub
                break
    if mounted:
        break

if mounted:
    print("dùng cache đã mount:", mounted)
    os.environ["LLDMMRI_CACHE_DIR"] = str(mounted)
    CACHE_DIR = mounted
    BUILD_NEEDED = False
else:
    print("không thấy cache E4 nào được mount -> sẽ build lại (~26 phút)")
    BUILD_NEEDED = True

In [ ]:
if BUILD_NEEDED:
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    rc = subprocess.run(
        [sys.executable, "-m", "src.preprocess.build_cache",
         "--config", "configs/preprocess_e4.yaml"],
        cwd=REPO,
    ).returncode
    assert rc == 0, "build cache thất bại"
    os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)
else:
    print("bỏ qua build")

## Cổng A ⚠️⚠️ — cache này có ĐÚNG là cache E4 không

Đây là cổng quan trọng nhất của notebook. Chạy CV trên cache của E1 hay E3 sẽ **không
báo lỗi gì cả** — nó chỉ lặng lẽ cho ra một bảng kết quả sai, và bảng đó trông vẫn
hợp lý. Ba khoá dưới đây là thứ phân biệt E4 với mọi cache trước đó.

In [ ]:
import json

meta = json.loads((Path(os.environ["LLDMMRI_CACHE_DIR"]) / "cache_meta.json").read_text("utf-8"))

EXPECTED = {
    "align_phases": "per_phase",          # <- khoá phân biệt E4 với E3
    "target_size": [112, 112, 32],        # <- khoá phân biệt E3/E4 với E0/E1
    "crop_mode": "lesion_tight",          # <- khoá phân biệt E1+ với E0
}
for key, want in EXPECTED.items():
    got = meta.get(key)
    assert got == want, f"cache SAI: {key} = {got!r}, cần {want!r}. Đây không phải cache E4."
assert meta["lesion_tight"]["source"] == "mask", "phải cắt theo mask, không phải bbox"

n_npz = len(list(Path(os.environ["LLDMMRI_CACHE_DIR"]).glob("*.npz")))
print("cache_meta khớp E4 ✓")
for key in ("crop_mode", "target_size", "align_phases", "git_commit"):
    print(f"  {key:>14}: {meta.get(key)}")
print(f"  {'số file .npz':>14}: {n_npz}")
assert n_npz >= 498, f"chỉ có {n_npz} ca, cần 498 — cache chưa build xong"

# PHẦN B — Train

## B1 — GPU và thời gian một epoch

In [ ]:
import time

import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "—")

SECONDS_PER_EPOCH = 46.0      # đo ở fold 1 của E4
EPOCHS = int(CFG["train"]["epochs"])
est_hours = SECONDS_PER_EPOCH * EPOCHS / 3600
print(f"\nước tính: {est_hours:.2f} h/fold × {len(FOLDS)} fold = {est_hours*len(FOLDS):.2f} h")
if BUILD_NEEDED:
    print("cộng thêm ~0.45h build cache")
assert est_hours * len(FOLDS) < 11.0, "quá sát trần 12h — bớt fold trong FOLDS đi"

## B2 — Chạy từng fold

Chạy lại **chính cell này** nếu session bị ngắt: `resume: true` nạp lại `last.pt`, và
fold nào đã đủ 300 epoch thì trả về ngay.

In [ ]:
from src.train.run import run_dir, train

results = {}
t_start = time.time()

for fold in FOLDS:
    elapsed_h = (time.time() - t_start) / 3600
    if elapsed_h + est_hours > 11.0:
        print(f"\nDỪNG: đã dùng {elapsed_h:.2f}h, chạy tiếp fold {fold} sẽ vượt trần 12h.")
        print("Fold còn lại để session sau — checkpoint đã lưu, không mất gì.")
        break

    print(f"\n{'='*70}\nFOLD {fold}  (đã dùng {elapsed_h:.2f}h)\n{'='*70}")
    results[fold] = train(CFG_PATH, fold_override=fold)
    print(f"fold {fold} xong: macro-F1 {results[fold].get('macro_f1', float('nan')):.4f}")

print(f"\ntổng thời gian: {(time.time()-t_start)/3600:.2f} h")

## B3 — Kết quả các fold đã có

Bảng này chỉ liệt kê fold đã chạy **trong cây output hiện tại**. Fold 1 nằm ở run E4
cũ nên sẽ không xuất hiện ở đây — gộp đủ 5 fold là việc làm ở máy local sau khi tải
về (xem B4).

In [ ]:
import numpy as np

OUT_ROOT = Path(os.environ["LLDMMRI_OUTPUT_DIR"])
rows = []
for d in sorted(OUT_ROOT.glob("fold*")):
    f = d / "metrics_best.json"
    if f.exists():
        m = json.loads(f.read_text("utf-8"))
        rows.append((d.name, m["fold"], m["epoch"], m["macro_f1"], m["cohen_kappa"], m["accuracy"]))

print(f"{'thư mục':<20}{'fold':>5}{'epoch':>7}{'macro-F1':>11}{'kappa':>9}{'acc':>8}")
print("-" * 60)
for name, fold, ep, f1, kp, ac in rows:
    print(f"{name:<20}{fold:>5}{ep:>7}{f1:>11.4f}{kp:>9.4f}{ac:>8.4f}")

if rows:
    f1s = np.array([r[3] for r in rows])
    print(f"\ntrung bình macro-F1 các fold trong session này: {f1s.mean():.4f} (±{f1s.std():.4f})")
print("\nĐỐI CHIẾU fold 1 (đã chạy trước): macro-F1 0.7001")
print("Fold khác nhau về độ khó — chênh ±0.05 giữa các fold là bình thường,")
print("con số dùng để báo cáo là bản GỘP OUT-OF-FOLD, không phải trung bình này.")

## B4 — Gói mang về

Chỉ lấy thứ nhẹ: `val_probs_best.npz` (xác suất từng ca — đủ để tính lại **mọi**
metric, calibration và selective ở máy local, không cần GPU), `metrics_best.json`,
`train_log.csv`, `config_used.json`.

**`best.pt` cũng lấy** — 5 checkpoint này *là* deep ensemble, thứ mà web app và phần
bất định epistemic sẽ dùng. Nếu output quá nặng thì bỏ `last.pt` (nó chỉ để resume).

In [ ]:
import shutil

PACK = Path("/kaggle/working/E4_cv_results")
shutil.rmtree(PACK, ignore_errors=True)
PACK.mkdir(parents=True)

KEEP = ["val_probs_best.npz", "metrics_best.json", "train_log.csv", "config_used.json", "best.pt"]
for d in sorted(OUT_ROOT.glob("fold*")):
    dst = PACK / d.name          # GIỮ NGUYÊN tên fold{N}_{hash} — src.eval.run cần nó
    dst.mkdir(parents=True, exist_ok=True)
    for name in KEEP:
        if (d / name).exists():
            shutil.copy2(d / name, dst / name)

shutil.copy2(Path(os.environ["LLDMMRI_CACHE_DIR"]) / "cache_meta.json", PACK / "cache_meta.json")

total = sum(f.stat().st_size for f in PACK.rglob("*") if f.is_file())
print(f"đã gói {PACK}: {total/2**20:.1f} MiB")
for f in sorted(PACK.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(PACK)}  {f.stat().st_size/2**20:.2f} MiB")

print("""
Ở MÁY LOCAL sau khi tải về:
  1. giải nén vào  runs/E4_cv_results/
  2. chép fold 1 vào cùng cây, giữ đúng quy ước tên thư mục:
         mkdir -p runs/E4_cv_results/fold1_4c2cf705
         cp runs/E4_per_phase_results/val_probs_best.npz runs/E4_cv_results/fold1_4c2cf705/
  3. python -m src.eval.run --run-dir runs/E4_cv_results
     -> bảng từng fold + GỘP OUT-OF-FOLD 394 ca kèm CI bootstrap
""")